# Punto 5 - Limpieza del conjunto MINEDUC

Este notebook se crea **desde cero** y cubre **solo el punto 5** del plan de limpieza (`PLAN_LIMPIEZA.md`).

Se aplica limpieza reproducible y se documenta cada inciso requerido:

- a) faltantes y marcadores de ausencia,
- b) tipos de dato,
- c) normalizacion de texto,
- d) consistencia de categorias,
- e) formatos,
- f) valores invalidos,
- g) duplicados exactos y parciales,
- h) consistencia entre variables,
- i) variables derivadas y su justificacion.

In [18]:
from __future__ import annotations

import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / 'data' / 'raw' / 'establecimientos_diversificado_guatemala.csv'
CAT_DIR = ROOT / 'data' / 'raw' / 'catalogos'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV_PATH = PROCESSED_DIR / 'establecimientos_diversificado_guatemala_limpio.csv'
DUPLICADOS_PATH = PROCESSED_DIR / 'duplicados_candidatos.csv'
LOG_CSV_PATH = PROCESSED_DIR / 'log_transformaciones.csv'
LOG_MD_PATH = PROCESSED_DIR / 'log_transformaciones.md'
CODEBOOK_PATH = PROCESSED_DIR / 'libro_codigos.md'

PLACEHOLDERS = {'', '--', 'N/A', 'NULL', '-', '.', 'SIN DATO', 'NA', 'N.D.'}
CATEGORICAL_UPPER = [
    'departamento', 'municipio', 'nivel', 'sector', 'area', 'status', 'modalidad',
    'jornada', 'plan', 'departamental', 'departamento_consulta',
    'nivel_consulta', 'sector_consulta', 'plan_consulta', 'modalidad_consulta'
]

df_raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
df = df_raw.copy()

cat_departamentos = pd.read_csv(CAT_DIR / 'departamentos.csv', dtype=str, keep_default_na=False)
cat_municipios = pd.read_csv(CAT_DIR / 'municipios.csv', dtype=str, keep_default_na=False)
cat_sectores = pd.read_csv(CAT_DIR / 'sectores.csv', dtype=str, keep_default_na=False)
cat_modalidades = pd.read_csv(CAT_DIR / 'modalidades.csv', dtype=str, keep_default_na=False)

print(f'Archivo crudo: {RAW_PATH}')
print(f'Filas x columnas (crudo): {df.shape}')

Archivo crudo: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\raw\establecimientos_diversificado_guatemala.csv
Filas x columnas (crudo): (11867, 26)


## Aplicacion del punto 5 (a-i)

**a) Valores faltantes y NA**  
Se normalizan marcadores (`''`, `--`, `N/A`, `NULL`, `-`, `.`, `Sin dato`, `SIN DATO`, `NA`, `N.D.`) a `NA` cuando representan ausencia. `SIN ESPECIFICAR` se conserva como categoria valida en `area`.

**b) Tipos de dato**  
Se carga con `dtype=str` para preservar codigos y telefonos. `fecha_extraccion` se convierte a `datetime` con UTC.

**c) Normalizacion de texto**  
Se eliminan espacios extremos, espacios multiples e invisibles. No se eliminan tildes ni signos con valor semantico.

**d) Consistencia de categorias**  
Se validan `departamento`, `municipio` (en par con departamento), `sector` y `modalidad` contra catalogos.

**e) Formatos**  
Se revisa formato de `codigo`, se normaliza `telefono` y se marca validez (`telefono_valido`). Se crea `distrito_formato_revisar`.

**f) Valores invalidos**  
Se reportan valores fuera de dominio o formato sin corregir por inferencia.

**g) Duplicados**  
Se reportan duplicados exactos y candidatos parciales por similitud de cadenas (no se eliminan automaticamente).

**h) Consistencia entre variables**  
Se marca discrepancia `departamental` vs `departamento` y consistencia entre `departamento_consulta` y su codigo.

**i) Variables derivadas**  
Se crean: `telefono_normalizado`, `telefono_valido`, `distrito_formato_revisar`, `departamental_difiere_departamento`.

In [19]:
def normalize_text(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    s = str(value)
    s = s.replace('\u00A0', ' ')
    s = re.sub(r'[\u200B-\u200D\uFEFF]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def to_missing(value: object, column: str) -> object:
    s = normalize_text(value)
    if pd.isna(s):
        return pd.NA
    upper = str(s).upper()
    if column == 'area' and upper == 'SIN ESPECIFICAR':
        return 'SIN ESPECIFICAR'
    if upper == 'SIN DATO' or upper in PLACEHOLDERS:
        return pd.NA
    return s


def normalize_phone(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    s = str(value)
    s = re.sub(r'[\s\-\(\)]', '', s)
    if s.startswith('+502'):
        s = s[4:]
    digits = re.sub(r'\D', '', s)
    return digits if digits else pd.NA


def fold_for_match(value: object) -> str:
    if pd.isna(value):
        return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


for col in df.columns:
    if col != 'fecha_extraccion':
        df[col] = df[col].apply(lambda v: to_missing(v, col))

for col in CATEGORICAL_UPPER:
    if col in df.columns:
        df[col] = df[col].astype('string').str.upper()

df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'], errors='coerce', utc=True)
df['departamento_consulta_codigo'] = df['departamento_consulta_codigo'].astype('string')

df['telefono_normalizado'] = df['telefono'].apply(normalize_phone)
df['telefono_valido'] = df['telefono_normalizado'].astype('string').str.fullmatch(r'\d{8}').fillna(False)

df['distrito_formato_revisar'] = (
    df['distrito'].notna() &
    ~df['distrito'].astype('string').str.fullmatch(r'\d{2}-\d{2}')
)
df['departamental_difiere_departamento'] = (
    df['departamental'].notna() &
    df['departamento'].notna() &
    (df['departamental'] != df['departamento'])
)

if 'municipio_consulta' in df.columns and df['municipio_consulta'].isna().all():
    df = df.drop(columns=['municipio_consulta'])

In [20]:
valid_departamentos = set(cat_departamentos['label'].str.upper())
valid_sector = set(cat_sectores['label'].str.upper())
valid_modalidad = set(cat_modalidades['label'].str.upper())
valid_municipio_pairs = set(
    zip(
        cat_municipios['departamento'].str.upper(),
        cat_municipios['municipio'].str.upper(),
    )
)
code_to_depto = dict(zip(cat_departamentos['value'], cat_departamentos['label'].str.upper()))

invalid_codigo = int((df['codigo'].notna() & ~df['codigo'].astype('string').str.fullmatch(r'\d{2}-\d{2}-\d{4}-\d{2}')).sum())
invalid_departamento = int((df['departamento'].notna() & ~df['departamento'].isin(valid_departamentos)).sum())
invalid_municipio_pair = int((df['departamento'].notna() & df['municipio'].notna() & ~pd.Series(list(zip(df['departamento'], df['municipio']))).isin(valid_municipio_pairs)).sum())
invalid_sector = int((df['sector'].notna() & ~df['sector'].isin(valid_sector)).sum())
invalid_modalidad = int((df['modalidad'].notna() & ~df['modalidad'].isin(valid_modalidad)).sum())
invalid_telefono = int((df['telefono_normalizado'].notna() & ~df['telefono_normalizado'].astype('string').str.fullmatch(r'\d{8}')).sum())

departamental_diff = int(df['departamental_difiere_departamento'].sum())
consulta_diff = int((df['departamento_consulta'].notna() & df['departamento'].notna() & (df['departamento_consulta'] != df['departamento'])).sum())
consulta_code_diff = int((df['departamento_consulta_codigo'].notna() & df['departamento_consulta'].notna() & (df['departamento_consulta_codigo'].map(code_to_depto) != df['departamento_consulta'])).sum())

duplicados_exactos = int(df.duplicated().sum())

dup_base = df[['codigo', 'municipio', 'establecimiento', 'direccion', 'telefono_normalizado']].copy()
dup_base['nombre_cmp'] = dup_base['establecimiento'].apply(fold_for_match)
dup_base['direccion_cmp'] = dup_base['direccion'].apply(fold_for_match)
dup_base = dup_base[dup_base['municipio'].notna() & dup_base['nombre_cmp'].ne('')]
dup_base['block_name'] = dup_base['nombre_cmp'].str[:12]

candidatos = []
for municipio, group in dup_base.groupby('municipio', dropna=True):
    for _, block in group.groupby('block_name'):
        rows = block.to_dict('records')
        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                a = rows[i]
                b = rows[j]
                if a['codigo'] == b['codigo']:
                    continue
                score_nombre = SequenceMatcher(None, a['nombre_cmp'], b['nombre_cmp']).ratio()
                score_direccion = SequenceMatcher(None, a['direccion_cmp'], b['direccion_cmp']).ratio() if a['direccion_cmp'] and b['direccion_cmp'] else 0.0
                same_phone = (
                    pd.notna(a['telefono_normalizado']) and
                    pd.notna(b['telefono_normalizado']) and
                    a['telefono_normalizado'] == b['telefono_normalizado']
                )
                if same_phone or (score_nombre >= 0.95) or (score_nombre >= 0.90 and score_direccion >= 0.75):
                    candidatos.append({
                        'codigo_1': a['codigo'],
                        'codigo_2': b['codigo'],
                        'municipio': municipio,
                        'score_nombre': round(score_nombre, 4),
                        'score_direccion': round(score_direccion, 4),
                        'telefono_igual': same_phone,
                        'decision': 'pendiente_revision_manual',
                    })

duplicados_candidatos = pd.DataFrame(candidatos).drop_duplicates()

log_rows = [
    {
        'inciso': 'a',
        'tema': 'Valores faltantes y NA',
        'accion': 'Marcadores de ausencia normalizados a NA sin tocar SIN ESPECIFICAR en area.',
        'registros_afectados': int(df.isna().sum().sum()),
        'justificacion': 'Uniformar ausencias para analisis reproducible y conservar categorias validas.',
    },
    {
        'inciso': 'b',
        'tema': 'Tipos de dato',
        'accion': 'Dataset cargado como texto; fecha_extraccion convertida a datetime UTC.',
        'registros_afectados': int(df['fecha_extraccion'].notna().sum()),
        'justificacion': 'Preservar codigos y telefonos; garantizar tipo temporal correcto.',
    },
    {
        'inciso': 'c',
        'tema': 'Normalizacion de texto',
        'accion': 'Trim, colapso de espacios y eliminacion de invisibles.',
        'registros_afectados': int(len(df)),
        'justificacion': 'Evitar diferencias espurias de escritura sin perder semantica.',
    },
    {
        'inciso': 'd',
        'tema': 'Consistencia de categorias',
        'accion': 'Validacion contra catalogos oficiales de departamento, municipio, sector y modalidad.',
        'registros_afectados': int(invalid_departamento + invalid_municipio_pair + invalid_sector + invalid_modalidad),
        'justificacion': 'Detectar categorias fuera de dominio sin corregir por inferencia.',
    },
    {
        'inciso': 'e',
        'tema': 'Formatos',
        'accion': 'Validacion de codigo, normalizacion de telefono y bandera de distrito.',
        'registros_afectados': int(invalid_codigo + invalid_telefono + df['distrito_formato_revisar'].sum()),
        'justificacion': 'Aislar formatos problematicos para revision transparente.',
    },
    {
        'inciso': 'f',
        'tema': 'Valores invalidos',
        'accion': 'Conteo de fuera de dominio en categorias, codigos y telefonos.',
        'registros_afectados': int(invalid_codigo + invalid_departamento + invalid_municipio_pair + invalid_sector + invalid_modalidad + invalid_telefono),
        'justificacion': 'Documentar invalidez sin alterar valores dudosos automaticamente.',
    },
    {
        'inciso': 'g',
        'tema': 'Registros duplicados',
        'accion': 'Busqueda de duplicados exactos y candidatos parciales con similitud de cadenas.',
        'registros_afectados': int(duplicados_exactos + len(duplicados_candidatos)),
        'justificacion': 'No se elimina automaticamente; se deja evidencia para decision manual.',
    },
    {
        'inciso': 'h',
        'tema': 'Consistencia entre variables',
        'accion': 'Revision de coherencia entre departamental, departamento y metadatos de consulta.',
        'registros_afectados': int(departamental_diff + consulta_diff + consulta_code_diff),
        'justificacion': 'Hacer visibles contradicciones entre columnas relacionadas.',
    },
    {
        'inciso': 'i',
        'tema': 'Variables derivadas',
        'accion': 'Creacion de telefono_normalizado, telefono_valido, distrito_formato_revisar y departamental_difiere_departamento.',
        'registros_afectados': int(len(df)),
        'justificacion': 'Permiten auditar calidad sin perder el dato original.',
    },
]

log_df = pd.DataFrame(log_rows)
log_df.to_csv(LOG_CSV_PATH, index=False, encoding='utf-8')

markdown_lines = [
    '# Log de transformaciones (Punto 5)',
    '',
    '| inciso | tema | accion | registros_afectados | justificacion |',
    '|---|---|---|---:|---|',
]
for _, row in log_df.iterrows():
    markdown_lines.append(
        f"| {row['inciso']} | {row['tema']} | {row['accion']} | {int(row['registros_afectados'])} | {row['justificacion']} |"
    )
LOG_MD_PATH.write_text('\n'.join(markdown_lines), encoding='utf-8')

derived_description = {
    'telefono_normalizado': 'Telefono con limpieza de formato para comparacion y validacion.',
    'telefono_valido': 'Bandera booleana: telefono_normalizado tiene 8 digitos.',
    'distrito_formato_revisar': 'Bandera booleana: distrito no cumple patron esperado NN-NN.',
    'departamental_difiere_departamento': 'Bandera booleana: departamental difiere de departamento.',
}

book = df.dtypes.astype(str).rename('tipo_dato').to_frame()
book['nulos'] = df.isna().sum()
book['es_derivada'] = book.index.isin(derived_description.keys())
book['descripcion'] = [derived_description.get(col, '') for col in book.index]

codebook_lines = [
    '# Libro de codigos - Limpieza Punto 5',
    '',
    'Variables derivadas y justificacion:',
    '- telefono_normalizado: estandariza telefono para validacion y deteccion de duplicados.',
    '- telefono_valido: identifica telefonos con estructura de 8 digitos.',
    '- distrito_formato_revisar: identifica distritos fuera del patron esperado para revision manual.',
    '- departamental_difiere_departamento: identifica posibles inconsistencias administrativas.',
    '',
    '| variable | tipo_dato | nulos | es_derivada | descripcion |',
    '|---|---|---:|---|---|',
]
for variable, row in book.iterrows():
    codebook_lines.append(
        f"| {variable} | {row['tipo_dato']} | {int(row['nulos'])} | {bool(row['es_derivada'])} | {row['descripcion']} |"
    )
CODEBOOK_PATH.write_text('\n'.join(codebook_lines), encoding='utf-8')

df.to_csv(CLEAN_CSV_PATH, index=False, encoding='utf-8')
duplicados_candidatos.to_csv(DUPLICADOS_PATH, index=False, encoding='utf-8')

resumen = pd.DataFrame([
    ('filas_limpias', len(df)),
    ('columnas_limpias', df.shape[1]),
    ('duplicados_exactos', duplicados_exactos),
    ('duplicados_parciales_candidatos', len(duplicados_candidatos)),
    ('invalid_codigo', invalid_codigo),
    ('invalid_departamento', invalid_departamento),
    ('invalid_municipio_pair', invalid_municipio_pair),
    ('invalid_sector', invalid_sector),
    ('invalid_modalidad', invalid_modalidad),
    ('invalid_telefono', invalid_telefono),
    ('departamental_vs_departamento_difiere', departamental_diff),
    ('departamento_consulta_vs_departamento_difiere', consulta_diff),
    ('codigo_departamento_consulta_inconsistente', consulta_code_diff),
], columns=['metrica', 'valor'])

print(f'CSV limpio: {CLEAN_CSV_PATH}')
print(f'Duplicados parciales: {DUPLICADOS_PATH}')
print(f'Log CSV: {LOG_CSV_PATH}')
print(f'Log MD: {LOG_MD_PATH}')
print(f'Libro de codigos: {CODEBOOK_PATH}')
resumen

CSV limpio: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\processed\establecimientos_diversificado_guatemala_limpio.csv
Duplicados parciales: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\processed\duplicados_candidatos.csv
Log CSV: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\processed\log_transformaciones.csv
Log MD: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\processed\log_transformaciones.md
Libro de codigos: C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\processed\libro_codigos.md


,metrica,valor
0,filas_limpias,11867
1,columnas_limpias,29
2,duplicados_exactos,0
3,duplicados_parciales_candidatos,13223
4,invalid_codigo,0
5,invalid_departamento,0
6,invalid_municipio_pair,0
7,invalid_sector,0
8,invalid_modalidad,0
9,invalid_telefono,248


## Justificacion final del punto 5

- Se limpio el dataset sin tocar el archivo crudo.
- Se documentaron todos los incisos (a-i) con acciones, conteos y justificacion.
- Los duplicados parciales se listan para revision manual; no se eliminan automaticamente.
- Las variables derivadas se incluyen en el libro de codigos con su utilidad.